# German Power Demand: A Comparative Forecasting Study (Version 2)

This notebook forecasts the national electricity load of Germany and benchmarks a
ladder of forecasting techniques against one another over a **two-year weekly
hold-out**. The workflow moves from raw hourly readings to weekly aggregates,
characterises the series statistically, and then contrasts naive baselines, a
seasonal ARIMA family, an ARIMA model augmented with weather/calendar drivers, a
**Gradient Boosting** regressor, and an hourly LSTM.

**What this version emphasises**

- A differencing decision that is defended with unit-root evidence rather than a
  blind AIC grid, because AIC is only comparable within a single differencing order.
- A seasonal `(P, D, Q)` search kept deliberately small given how few full annual
  cycles the training window contains.
- Whitened (burn-in-trimmed) residual checks for the state-space models.
- A **genuine recursive** multi-step forecast for the machine-learning model so it
  is comparable with SARIMA, plus a separately-labelled one-step variant.
- Rolling versus open-loop LSTM errors reported with unambiguous labels.
- RMSE, MAE and MAPE for every model together with an explicit forecast-type tag.

## 0. Environment, styling and shared helpers

Everything the rest of the notebook relies on is set up once here: the plotting
theme and colour palette, the reproducibility seed, and two small scoring helpers.
The error metrics are computed directly from NumPy so the notebook does not lean on
any external metric utilities.

In [ ]:
%pip install -q holidays
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cycler

# ---- visual identity (deliberately purple / earth-toned) -------------------
plt.style.use('bmh')
HUES = {
    'violet': '#6a4c93',
    'brick':  '#9e2a2b',
    'olive':  '#6b8e23',
    'slate':  '#343a40',
    'teal':   '#1b7f79',
    'amber':  '#bc8a00',
    'silver': '#adb5bd',
}
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f4f1f7',
    'axes.prop_cycle': cycler(color=[HUES['violet'], HUES['brick'], HUES['olive'],
                                     HUES['teal'], HUES['amber'], HUES['silver']]),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10.5,
})

SEED = 2024
np.random.seed(SEED)


def accuracy_metrics(actual, predicted):
    """RMSE, MAE and MAPE for a single forecast, returned as a dictionary."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    gap = actual - predicted
    return {
        'rmse': float(np.sqrt(np.mean(gap ** 2))),
        'mae': float(np.mean(np.abs(gap))),
        'mape': float(np.mean(np.abs(gap / actual)) * 100.0),
    }


def root_mse(actual, predicted):
    """Stand-alone RMSE, used mainly by the neural-network section."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

## 1. Reading and reshaping the load series

The source is the Open Power System Data 60-minute file for Germany. We keep the
actual-load column reported to ENTSO-E, restrict the window to 2015 onward, and
collapse the hourly signal into **daily** and **weekly** means. Weekly averaging
smooths out working-day/weekend noise and brings the annual heating cycle to the
foreground, which is what the classical models exploit.

In [ ]:
CSV_PATH = '/kaggle/input/datasets/rishiande/german/opsd_60min_raw.csv'
opsd_hourly = pd.read_csv(CSV_PATH, parse_dates=['utc_timestamp'], index_col='utc_timestamp')
print(f'Raw hourly records loaded: {len(opsd_hourly):,}')

In [ ]:
SOURCE_COL = 'DE_load_actual_entsoe_transparency'
de_power = opsd_hourly[[SOURCE_COL]].rename(columns={SOURCE_COL: 'mw'}).copy()
de_power = de_power.loc['2015-01-01':'2020-10-31'].dropna()
print('Coverage    :', de_power.index.min(), '->', de_power.index.max())
print('Hourly kept :', f'{len(de_power):,}')

In [ ]:
load_by_day = de_power['mw'].resample('D').mean()
load_by_week = de_power['mw'].resample('W').mean()
print('Weekly points :', len(load_by_week))
print('Weekly gaps?  :', bool(load_by_week.isna().any()))

In [ ]:
summary = pd.concat([load_by_day.describe().rename('daily'),
                     load_by_week.describe().rename('weekly')], axis=1)
print(summary.round(1).to_string())

### Visual overview

The daily trace is drawn thin, with a robust 30-day rolling **median** riding on top
to expose the trend without letting outliers pull it around. An arrow flags the 2020
lockdown demand drop. The weekly panel shows the aggregated series as markers joined
by a light line, with the long-run average as a reference.

In [ ]:
fig, (top, bot) = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'hspace': 0.3})

daily_med = load_by_day.rolling(30, center=True).median()
top.plot(load_by_day.index, load_by_day, color=HUES['silver'], lw=0.7, label='Daily mean')
top.plot(daily_med.index, daily_med, color=HUES['brick'], lw=2.2, label='30-day rolling median')
dip_y = load_by_day.loc['2020-03-15':'2020-05-31'].min()
top.annotate('COVID-19 lockdown', xy=(pd.Timestamp('2020-04-15'), dip_y),
             xytext=(pd.Timestamp('2018-05-01'), load_by_day.min()),
             arrowprops=dict(arrowstyle='->', color=HUES['slate']),
             fontsize=10, color=HUES['slate'])
top.set(title='German daily electricity demand (2015-2020)', ylabel='MW')
top.legend(frameon=True)

bot.plot(load_by_week.index, load_by_week, color=HUES['violet'], lw=1.0,
         marker='o', ms=2.5, label='Weekly mean')
bot.axhline(load_by_week.mean(), color=HUES['teal'], ls=(0, (6, 3)), lw=1.4,
            label=f'Overall mean {load_by_week.mean():,.0f} MW')
bot.set(title='Weekly aggregated demand', ylabel='MW', xlabel='Date')
bot.legend(frameon=True)
plt.tight_layout()
plt.show()

Demand climbs every winter and relaxes in summer, a textbook annual cycle. To make
that structure explicit we run an additive decomposition at `period=52` and draw the
four components in a stacked layout (the residual is shown as a scatter to make the
irregular component easier to read). The centred moving-average trend cannot be
evaluated near the two ends of the sample, so the residual is undefined there.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

breakdown = seasonal_decompose(load_by_week, model='additive', period=52)
parts = [('Observed', breakdown.observed, HUES['slate']),
         ('Trend', breakdown.trend, HUES['brick']),
         ('Seasonal', breakdown.seasonal, HUES['olive']),
         ('Residual', breakdown.resid, HUES['violet'])]

fig, axz = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
for ax, (name, comp, col) in zip(axz, parts):
    if name == 'Residual':
        ax.scatter(comp.index, comp, s=8, color=col, alpha=0.7)
        ax.axhline(0, color=HUES['slate'], lw=0.8)
    else:
        ax.plot(comp.index, comp, color=col, lw=1.6)
        ax.fill_between(comp.index, comp, comp.min(), color=col, alpha=0.08)
    ax.set_ylabel(name)
axz[0].set_title('Additive decomposition of weekly demand (annual period = 52)')
axz[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()
print('Residual NaNs from the centred moving average:', int(breakdown.resid.isna().sum()))

## 2. Stationarity and the differencing decision

We probe the level series and two differenced versions with the Augmented
Dickey-Fuller (ADF) and KPSS tests, which look at stationarity from opposite null
hypotheses, and we inspect the correlation structure with ACF/PACF plots.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf


def unit_root_probe(x, tag):
    x = x.dropna()
    a_stat, a_p = adfuller(x)[:2]
    k_stat, k_p = kpss(x, regression='c', nlags='auto')[:2]
    print(f'[{tag}]')
    print(f'   ADF : stat={a_stat:8.3f}  p={a_p:6.4f}  -> '
          f"{'stationary' if a_p <= 0.05 else 'unit root'}")
    print(f'   KPSS: stat={k_stat:8.3f}  p={k_p:6.4f}  -> '
          f"{'non-stationary' if k_p < 0.05 else 'level stationary'}")


for tag, ser in [('level', load_by_week),
                 ('regular difference (1)', load_by_week.diff()),
                 ('seasonal difference (52)', load_by_week.diff(52))]:
    unit_root_probe(ser, tag)

In [ ]:
fig, axz = plt.subplots(2, 2, figsize=(13.5, 7.5))
plot_acf(load_by_week.dropna(), ax=axz[0, 0], lags=104, title='ACF - level series')
plot_pacf(load_by_week.dropna(), ax=axz[1, 0], lags=52, method='ywm', title='PACF - level series')
plot_acf(load_by_week.diff().dropna(), ax=axz[0, 1], lags=104, title='ACF - first difference')
plot_pacf(load_by_week.diff().dropna(), ax=axz[1, 1], lags=52, method='ywm', title='PACF - first difference')
plt.tight_layout()
plt.show()

**Reading the evidence.** ADF rejects the unit-root null on the raw weekly series
and KPSS does not reject level stationarity, so the level is already close to
mean-stationary. Yet the ACF decays slowly and spikes near lag 52, and the
decomposition shows a strong annual wave. For a *seasonal* ARIMA we therefore take
one seasonal difference (`D = 1`, `s = 52`) to strip the yearly pattern and one
regular difference (`d = 1`) to remove the leftover local drift. A second regular
difference (`d = 2`) is rejected: the series is already stationary after one
difference, so differencing again would inflate variance and inject a spurious
MA term. The SARIMA selection below confirms this numerically.

## 3. Naive baselines

Any serious model has to beat simple heuristics. We reserve the final **104 weeks
(two years)** as an untouched hold-out and score four classical forecasts on it:
the historical mean, last-value carry-forward, a seasonal replay of the previous
year, and a linear drift line.

In [ ]:
HOLDOUT = 104
fit_wk = load_by_week.iloc[:-HOLDOUT]
oos_wk = load_by_week.iloc[-HOLDOUT:]
idx = oos_wk.index
print(f'Training weeks : {len(fit_wk):3d}  (through {fit_wk.index[-1].date()})')
print(f'Hold-out weeks : {len(oos_wk):3d}  (through {oos_wk.index[-1].date()})')

In [ ]:
SEASON = 52

flat_mean = pd.Series(fit_wk.mean(), index=idx)
carry_forward = pd.Series(fit_wk.iloc[-1], index=idx)
cycle = fit_wk.iloc[-SEASON:].to_numpy()
seasonal_replay = pd.Series(np.resize(cycle, HOLDOUT), index=idx)
step = (fit_wk.iloc[-1] - fit_wk.iloc[0]) / (len(fit_wk) - 1)
linear_drift = pd.Series(fit_wk.iloc[-1] + step * np.arange(1, HOLDOUT + 1), index=idx)

baseline_bank = {'Historical mean': flat_mean, 'Last value': carry_forward,
                 'Seasonal replay': seasonal_replay, 'Linear drift': linear_drift}
print('--- baseline scores ---')
for name, series in baseline_bank.items():
    m = accuracy_metrics(oos_wk, series)
    print(f"{name:16s} RMSE={m['rmse']:8.1f}  MAE={m['mae']:8.1f}  MAPE={m['mape']:5.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 4.8))
ax.plot(idx, oos_wk, color=HUES['slate'], lw=2.4, label='Actual (hold-out)')
line_styles = {
    'Seasonal replay': dict(color=HUES['brick'], ls='-', lw=1.8),
    'Historical mean': dict(color=HUES['olive'], ls=(0, (4, 2))),
    'Linear drift': dict(color=HUES['teal'], ls=(0, (1, 1))),
    'Last value': dict(color=HUES['amber'], ls=(0, (5, 1))),
}
for name, kw in line_styles.items():
    ax.plot(idx, baseline_bank[name], label=name, **kw)
ax.set(title='Classical baselines across the two-year hold-out', ylabel='MW', xlabel='Date')
ax.legend(ncol=2, frameon=True)
plt.tight_layout()
plt.show()

## 4. Seasonal ARIMA

The brief asks for a full sweep of `p in [0,6]`, `d in [0,2]`, `q in [0,6]` (147
non-seasonal combinations). We run it, but selection is handled with care:

**AIC only compares within a fixed `d`.** The likelihood is evaluated on the
`d`-times-differenced series, so a `d = 2` model is scored on a shorter, differently
scaled series than a `d = 1` model. Ranking them on raw AIC is invalid, which is
exactly why a naive grid can appear to favour over-differenced models.

The procedure below therefore: (1) screens all 147 orders quickly and records AIC and
convergence; (2) prints the raw grid minimum with a caveat; (3) fixes `d = 1` from the
unit-root evidence; (4) refits the leading `d = 1` orders *exactly* so their AIC shares
a common basis and checks residual whiteness; (5) applies a parsimony rule; and (6)
searches a compact seasonal space before reporting full diagnostics.

In [ ]:
import itertools
import time
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from joblib import Parallel, delayed

M = 52
order_space = list(itertools.product(range(7), range(3), range(7)))
SEED_SEASONAL = (1, 1, 1, M)
print(f'Non-seasonal orders to screen: {len(order_space)}  (p=[0,6], d=[0,2], q=[0,6])')


def probe_aic(triple, seasonal, y):
    """Fast AIC probe with simple differencing. Only valid to rank orders sharing one d."""
    try:
        fitted = SARIMAX(y, order=triple, seasonal_order=seasonal,
                         enforce_invertibility=False, enforce_stationarity=False,
                         simple_differencing=True).fit(disp=False, method='lbfgs', maxiter=60)
        conv = bool((fitted.mle_retvals or {}).get('converged', False))
        return {'order': triple, 'd': triple[1], 'aic': fitted.aic, 'converged': conv}
    except Exception:
        return {'order': triple, 'd': triple[1], 'aic': np.inf, 'converged': False}


clock = time.time()
probe_rows = Parallel(n_jobs=-1)(delayed(probe_aic)(t, SEED_SEASONAL, fit_wk) for t in order_space)
probe_tbl = pd.DataFrame(probe_rows)
print(f'Screen finished in {time.time() - clock:.1f}s')

In [ ]:
grid_min = probe_tbl.sort_values('aic').iloc[0]
print('Unconstrained grid minimum :', tuple(grid_min['order']),
      f"(AIC={grid_min['aic']:.2f}, d={int(grid_min['order'][1])})")
print('Caveat: AIC lives on the d-differenced series and is NOT comparable across d.')
print('        We fix d from the unit-root evidence and compare exact refits within it.\n')

D_FIXED = 1


def whitened_resid(fitted, order, seasonal):
    """Burn-in-trimmed standardized residuals, with a raw-residual fallback."""
    warmup = order[1] + seasonal[1] * seasonal[3]
    try:
        arr = np.asarray(fitted.standardized_forecasts_error)
        arr = arr[0] if np.ndim(arr) > 1 else arr
    except Exception:
        arr = np.asarray(fitted.resid)
    trimmed = pd.Series(arr).replace([np.inf, -np.inf], np.nan)
    return trimmed.iloc[warmup:].dropna()


def refit_full(order, seasonal, y):
    """Exact ML fit (no simple differencing) plus a Ljung-Box p on whitened residuals."""
    fitted = SARIMAX(y, order=order, seasonal_order=seasonal,
                     enforce_invertibility=False, enforce_stationarity=False
                     ).fit(disp=False, method='lbfgs', maxiter=350)
    w = whitened_resid(fitted, order, seasonal)
    lag = min(20, max(1, len(w) - 2))
    lb_p = acorr_ljungbox(w, lags=[lag], return_df=True)['lb_pvalue'].iloc[0]
    return fitted, bool((fitted.mle_retvals or {}).get('converged', False)), lb_p


shortlist = (probe_tbl[(probe_tbl['d'] == D_FIXED) & probe_tbl['converged']]
             .sort_values('aic').head(10)['order'].tolist())
if not shortlist:
    shortlist = probe_tbl[probe_tbl['d'] == D_FIXED].sort_values('aic').head(10)['order'].tolist()
if not shortlist:
    shortlist = [(0, D_FIXED, 1), (1, D_FIXED, 1)]

exact_rows = []
for order in shortlist:
    try:
        fitted, conv, lb_p = refit_full(order, SEED_SEASONAL, fit_wk)
        exact_rows.append({'order': order, 'aic': fitted.aic, 'bic': fitted.bic,
                           'converged': conv, 'LB_p': round(lb_p, 4)})
    except Exception:
        exact_rows.append({'order': order, 'aic': np.inf, 'bic': np.inf,
                           'converged': False, 'LB_p': np.nan})
exact_tbl = pd.DataFrame(exact_rows)
exact_tbl = exact_tbl[np.isfinite(exact_tbl['aic'])].sort_values('aic').reset_index(drop=True)
if len(exact_tbl) == 0:
    exact_tbl = pd.DataFrame([{'order': (0, D_FIXED, 1), 'aic': np.nan, 'bic': np.nan,
                               'converged': False, 'LB_p': np.nan}])
print(f'Exact refits within d={D_FIXED} (AIC now on a common basis, seasonal={SEED_SEASONAL}):')
print(exact_tbl.to_string(index=False))

challenger = tuple(int(v) for v in grid_min['order'])
try:
    cf, cconv, clb = refit_full(challenger, SEED_SEASONAL, fit_wk)
    print(f'\nGrid-min {challenger} refit exactly -> AIC={cf.aic:.2f}, converged={cconv}, LB p={clb:.4f}')
    if challenger[1] >= 2:
        print('   d>=2 over-differences a series already stationary at d=1 -> rejected.')
except Exception as err:
    print(f'\nGrid-min {challenger} unstable on exact refit ({type(err).__name__}) -> rejected.')

In [ ]:
# Parsimony (Burnham & Anderson): orders within 2 AIC units are statistically tied,
# so prefer the one with the fewest AR + MA terms.
floor = exact_tbl['aic'].min()
tie_band = exact_tbl[exact_tbl['aic'] <= floor + 2.0]
if len(tie_band) == 0:
    tie_band = exact_tbl.head(1)
arima_order = min(tie_band['order'], key=lambda o: (o[0] + o[2], o[0]))
print(f'Lowest-AIC order at d={D_FIXED} : {exact_tbl["order"].iloc[0]}  (AIC={floor:.2f})')
print(f'Tied within 2 AIC units       : {list(tie_band["order"])}')
print(f'Most parsimonious selection   : {arima_order}')

### Seasonal component

The seasonal **period** is fixed at `s = 52` because weekly demand repeats annually.
The seasonal **orders** `(P, D, Q)` are searched, not assumed, but the space is kept
small (`P, Q in {0, 1}`, `D = 1`) for a concrete reason: with roughly 197 training
weeks and `D = 1` consuming 52 of them, there are fewer than three complete annual
cycles from which to estimate lag-52 behaviour, so a wider seasonal grid would fit
noise. Selection is by AIC among converged candidates.

In [ ]:
seasonal_candidates = [(P, 1, Q, M) for P in (0, 1) for Q in (0, 1)]
seas_rows = []
for so in seasonal_candidates:
    try:
        fitted, conv, lb_p = refit_full(arima_order, so, fit_wk)
        seas_rows.append({'seasonal': so, 'aic': fitted.aic, 'bic': fitted.bic,
                          'converged': conv, 'LB_p': round(lb_p, 4)})
    except Exception:
        seas_rows.append({'seasonal': so, 'aic': np.inf, 'bic': np.inf,
                          'converged': False, 'LB_p': np.nan})
seas_tbl = pd.DataFrame(seas_rows)
seas_tbl = seas_tbl[np.isfinite(seas_tbl['aic'])].sort_values('aic').reset_index(drop=True)
if len(seas_tbl) == 0:
    seas_tbl = pd.DataFrame([{'seasonal': SEED_SEASONAL, 'aic': np.nan, 'bic': np.nan,
                              'converged': False, 'LB_p': np.nan}])
print('Seasonal (P, D, Q, s) comparison:')
print(seas_tbl.to_string(index=False))
seas_order = seas_tbl['seasonal'].iloc[0]
print(f'\nSelected seasonal order: {seas_order}')

In [ ]:
import scipy.stats as st

sarima_fit = SARIMAX(fit_wk, order=arima_order, seasonal_order=seas_order,
                     enforce_invertibility=False, enforce_stationarity=False
                     ).fit(disp=False, method='lbfgs', maxiter=500)

resid_w = whitened_resid(sarima_fit, arima_order, seas_order)
shapiro_p = st.shapiro(resid_w)[1]
lb_lags = [l for l in (10, 20, 52) if l < len(resid_w)]
lb_report = acorr_ljungbox(resid_w, lags=lb_lags, return_df=True)

rule = '=' * 60
print(rule)
print(f'FINAL SARIMA  ->  {arima_order} x {seas_order}')
print(rule)
print(f'AIC / BIC          : {sarima_fit.aic:.2f} / {sarima_fit.bic:.2f}')
print(f'Optimiser converged: {bool(sarima_fit.mle_retvals.get("converged", False))}')
print(f'Shapiro-Wilk p     : {shapiro_p:.3f} '
      f"({'Gaussian' if shapiro_p > 0.05 else 'non-Gaussian'} residuals)")
print('Ljung-Box on whitened residuals (burn-in removed):')
print(lb_report.to_string())
print('-' * 60)
print('Why not the raw grid minimum:')
print(f' * cross-d AIC is invalid, so {challenger} (d={challenger[1]}) is not a valid overall winner;')
print(' * unit-root tests show stationarity at d=1, and d=2 over-differences;')
print(' * within d=1 the leaders tie inside 2 AIC units, so the simplest order wins,')
print('   and the seasonal search trims any redundant seasonal term.')

### Residual diagnostics

Two views of the whitened residuals: their autocorrelation (should look like noise)
and a normal quantile-quantile plot (points on the diagonal imply Gaussian errors).

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.2))
plot_acf(resid_w, ax=axL, lags=52, title='SARIMA whitened residuals - ACF')
st.probplot(resid_w, dist='norm', plot=axR)
axR.get_lines()[0].set(color=HUES['brick'], markersize=3)
axR.get_lines()[1].set_color(HUES['slate'])
axR.set_title('SARIMA residuals - normal Q-Q')
plt.tight_layout()
plt.show()

**Interpretation.** Some autocorrelation typically survives at longer lags, so the
residuals are not perfect white noise: the seasonal structure is captured, but
holiday effects and the 2020 shock leave traces a purely seasonal model cannot encode.
The Q-Q plot bends away from the diagonal in the tails, matching the Shapiro-Wilk
rejection of normality. The practical consequence is that the Gaussian 95% intervals
are **approximate** and may under-cover around irregular periods, which is why we never
over-claim SARIMA accuracy and benchmark everything against the seasonal replay.

In [ ]:
sarima_bundle = sarima_fit.get_forecast(steps=HOLDOUT)
sarima_fc_mean = sarima_bundle.predicted_mean
sarima_fc_mean.index = idx
sarima_band = sarima_bundle.conf_int(alpha=0.05)
sarima_band.index = idx

fig, ax = plt.subplots(figsize=(12.5, 4.8))
ax.plot(fit_wk.index[-80:], fit_wk.iloc[-80:], color=HUES['silver'], lw=1.2, label='Recent training')
ax.plot(idx, oos_wk, color=HUES['slate'], lw=2.2, label='Actual')
ax.plot(idx, sarima_fc_mean, color=HUES['brick'], lw=2.0, ls=(0, (5, 1)), label='SARIMA mean')
ax.fill_between(idx, sarima_band.iloc[:, 0], sarima_band.iloc[:, 1],
                color=HUES['brick'], alpha=0.15, label='95% interval')
ax.set(title=f'SARIMA {arima_order}x{seas_order} forecast vs actual', ylabel='MW', xlabel='Date')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

sarima_scores = accuracy_metrics(oos_wk, sarima_fc_mean)
print(f"SARIMA  RMSE={sarima_scores['rmse']:.1f}  MAE={sarima_scores['mae']:.1f}  MAPE={sarima_scores['mape']:.2f}%")

## 5. Building exogenous drivers

Before the covariate models, we assemble the external regressors in one place:
Berlin 2 m temperature (Open-Meteo archive), its square (to capture the U-shaped
heating/cooling response), a one-week temperature lag, and a German public-holiday
flag. Because the hold-out uses *observed* future temperature, any model that consumes
these is a **conditional** forecast rather than a fully operational one.

In [ ]:
import requests
import holidays

API = ('https://archive-api.open-meteo.com/v1/archive?latitude=52.52&longitude=13.41'
       '&start_date=2015-01-01&end_date=2020-09-30&hourly=temperature_2m&timezone=UTC')
payload = requests.get(API, timeout=60).json()
berlin_temp = pd.Series(payload['hourly']['temperature_2m'],
                        index=pd.to_datetime(payload['hourly']['time']), name='temp')
berlin_temp.index = berlin_temp.index.tz_localize('UTC')
temp_wk = (berlin_temp.resample('W').mean()
           .reindex(load_by_week.index).interpolate().bfill().ffill())

cal = holidays.Germany(years=range(2015, 2021))


def week_has_holiday(week_close):
    span = pd.date_range(end=week_close, periods=7)
    return int(any(day in cal for day in span))


holiday_flag_wk = pd.Series([week_has_holiday(d) for d in load_by_week.index],
                            index=load_by_week.index)

drivers = pd.DataFrame({
    'temp': temp_wk,
    'temp_sq': temp_wk ** 2,
    'temp_prev': temp_wk.shift(1).bfill(),
    'holiday': holiday_flag_wk,
})
print('Driver columns :', list(drivers.columns))
print('Null check     :', drivers.isna().sum().to_dict())

drivers_fit = drivers.iloc[:-HOLDOUT]
drivers_oos = drivers.iloc[-HOLDOUT:]

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
hb = ax.hexbin(drivers['temp'], load_by_week, gridsize=24, cmap='BuPu', mincnt=1)
fig.colorbar(hb, ax=ax, label='Week count')
coef = np.polyfit(drivers['temp'], load_by_week, 2)
grid_t = np.linspace(drivers['temp'].min(), drivers['temp'].max(), 120)
ax.plot(grid_t, np.polyval(coef, grid_t), color=HUES['brick'], lw=2.6, label='Quadratic trend')
ax.set(title='Temperature vs weekly demand (density hexbin)',
       xlabel='Weekly mean temperature (C)', ylabel='Weekly mean load (MW)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. SARIMAX with weather and calendar drivers

We refit the selected ARIMA structure with the driver matrix attached. The squared
temperature term lets the linear model approximate the U-shaped demand curve, the lag
captures delayed weather response, and the holiday flag captures predictable dips.

In [ ]:
sarimax_fit = SARIMAX(fit_wk, exog=drivers_fit, order=arima_order, seasonal_order=seas_order,
                      enforce_invertibility=False, enforce_stationarity=False
                      ).fit(disp=False, method='lbfgs', maxiter=500)

sarimax_bundle = sarimax_fit.get_forecast(steps=HOLDOUT, exog=drivers_oos)
sarimax_fc_mean = sarimax_bundle.predicted_mean
sarimax_fc_mean.index = idx
sarimax_band = sarimax_bundle.conf_int(alpha=0.05)
sarimax_band.index = idx

sarimax_scores = accuracy_metrics(oos_wk, sarimax_fc_mean)
print(f"SARIMAX RMSE={sarimax_scores['rmse']:.1f}  MAE={sarimax_scores['mae']:.1f}  MAPE={sarimax_scores['mape']:.2f}%")
print(f"SARIMA  RMSE={sarima_scores['rmse']:.1f}  (baseline without drivers)")

fig, ax = plt.subplots(figsize=(12.5, 4.8))
ax.plot(idx, oos_wk, color=HUES['slate'], lw=2.2, label='Actual')
ax.plot(idx, sarimax_fc_mean, color=HUES['olive'], lw=2.0, label='SARIMAX mean')
ax.fill_between(idx, sarimax_band.iloc[:, 0], sarimax_band.iloc[:, 1],
                color=HUES['olive'], alpha=0.15, label='95% interval')
ax.set(title='SARIMAX (temperature + holiday) forecast vs actual', ylabel='MW', xlabel='Date')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
sarimax_resid = whitened_resid(sarimax_fit, arima_order, seas_order)
fig, ax = plt.subplots(figsize=(11, 3.6))
plot_acf(sarimax_resid, ax=ax, lags=52, title='SARIMAX whitened residuals - ACF')
plt.tight_layout()
plt.show()
sarimax_lags = [l for l in (10, 20, 52) if l < len(sarimax_resid)]
print('Ljung-Box (SARIMAX whitened residuals):')
print(acorr_ljungbox(sarimax_resid, lags=sarimax_lags, return_df=True).to_string())

**Did the drivers help?** Compare the SARIMAX RMSE above with the pure SARIMA RMSE.
The covariates add explanatory value, but they are not fully known at the forecast
origin (temperature needs a weather forecast; holidays are deterministic), so the
result stays a conditional forecast.

## 7. Gradient Boosting regressor

For the feature-based model we use **Gradient Boosting**, which grows an additive
ensemble of shallow trees where each tree corrects the residuals of the ensemble so
far. Compared with a bagged forest it usually needs shallower trees and a learning
rate, and it tends to squeeze more signal out of a modest feature set.

**Leakage control.** Every lag/rolling feature looks only backward: `load_lag =
shift(1)`, `load_yearlag = shift(52)`, `temp_lag = shift(1)`, `temp_ma4 =
rolling(4).mean()` (trailing). Current-week temperature and holiday are treated as
known under the same conditional assumption as SARIMAX.

**Fair evaluation.** If the previous-week load feature is filled with the *actual*
value at test time, the model is secretly a one-step forecaster and is not comparable
with SARIMA. We therefore report two forecasts: a **recursive multi-step** one (the
model feeds its own predictions back into the lag features past the origin) and a
**one-step** reference that consumes actual lags.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

design = pd.DataFrame(index=load_by_week.index)
design['woy'] = load_by_week.index.isocalendar().week.astype(int).to_numpy()
design['mon'] = load_by_week.index.month
design['temp_c'] = drivers['temp']
design['hol'] = drivers['holiday']
design['temp_lag'] = drivers['temp'].shift(1)
design['temp_ma4'] = drivers['temp'].rolling(4).mean()
design['load_lag'] = load_by_week.shift(1)
design['load_yearlag'] = load_by_week.shift(52)
design['y'] = load_by_week.to_numpy()
design = design.dropna()

PREDICTORS = [c for c in design.columns if c != 'y']
design_fit = design.iloc[:-HOLDOUT]
design_oos = design.iloc[-HOLDOUT:]
Xtr, ytr = design_fit[PREDICTORS], design_fit['y']
Xte, yte = design_oos[PREDICTORS], design_oos['y']
print(f'Design matrix ready: {len(design_fit)} training weeks, {len(PREDICTORS)} predictors.')
print('Predictors:', PREDICTORS)

### Hyperparameter search

Boosting is sensitive to the interplay of learning rate, tree count and tree depth, so
rather than guess we tune it on an **inner validation slice** - the last 52 weeks of the
training window, held back from fitting. A compact grid is scored by one-step validation
RMSE and the winning combination is then refitted on the full training window. This keeps
the tuning honest (the two-year hold-out is never touched) while giving the booster a
fair chance to express its capacity.

In [ ]:
# Inner validation slice (last 52 training weeks) drives the hyperparameter choice;
# the two-year hold-out stays untouched.
from itertools import product as combos

val_span = 52
inner_tr = design_fit.iloc[:-val_span]
inner_val = design_fit.iloc[-val_span:]
Xit, yit = inner_tr[PREDICTORS], inner_tr['y']
Xiv, yiv = inner_val[PREDICTORS], inner_val['y']

search_log = []
for trees, rate, depth in combos((300, 500), (0.03, 0.05, 0.1), (2, 3)):
    probe = GradientBoostingRegressor(n_estimators=trees, learning_rate=rate, max_depth=depth,
                                      subsample=0.85, min_samples_leaf=6, random_state=SEED)
    probe.fit(Xit, yit)
    score = accuracy_metrics(yiv, probe.predict(Xiv))
    search_log.append({'trees': trees, 'rate': rate, 'depth': depth,
                       'val RMSE': round(score['rmse'], 1)})
gb_search = pd.DataFrame(search_log).sort_values('val RMSE').reset_index(drop=True)
print(gb_search.to_string(index=False))

pick = gb_search.iloc[0]
gb_params = dict(n_estimators=int(pick['trees']), learning_rate=float(pick['rate']),
                 max_depth=int(pick['depth']), subsample=0.85, min_samples_leaf=6, random_state=SEED)
print('\nSelected booster hyperparameters:', gb_params)

In [ ]:
gb_engine = GradientBoostingRegressor(**gb_params)
gb_engine.fit(Xtr, ytr)
print(f'Gradient Boosting refitted on all {len(Xtr)} training weeks with the tuned settings.')

In [ ]:
# (a) one-step: each week is handed the ACTUAL previous-week load.
gb_onestep = pd.Series(gb_engine.predict(Xte), index=yte.index)

# (b) recursive multi-step from a single origin (end of training). The load lags are
#     progressively filled with the model's OWN predictions once we pass known history,
#     which makes this comparable to SARIMA/SARIMAX. Temperature/holiday stay observed.
axis = load_by_week.index
origin = len(fit_wk)
history = list(load_by_week.iloc[:origin].to_numpy())
temp_ref = drivers['temp']
hol_ref = drivers['holiday']

roll_out = []
for h in range(HOLDOUT):
    pos = origin + h
    when = axis[pos]
    feat_row = {
        'woy': int(when.isocalendar()[1]),
        'mon': when.month,
        'temp_c': temp_ref.iloc[pos],
        'hol': hol_ref.iloc[pos],
        'temp_lag': temp_ref.iloc[pos - 1],
        'temp_ma4': temp_ref.iloc[pos - 3:pos + 1].mean(),
        'load_lag': history[pos - 1],
        'load_yearlag': history[pos - 52],
    }
    yhat = float(gb_engine.predict(pd.DataFrame([feat_row])[PREDICTORS])[0])
    roll_out.append(yhat)
    history.append(yhat)
gb_recursive = pd.Series(roll_out, index=idx)

gb_one_scores = accuracy_metrics(yte, gb_onestep)
gb_rec_scores = accuracy_metrics(oos_wk, gb_recursive)
print(f"GB one-step   RMSE={gb_one_scores['rmse']:8.1f}  MAE={gb_one_scores['mae']:8.1f}  MAPE={gb_one_scores['mape']:5.2f}%   (actual lag-1)")
print(f"GB recursive  RMSE={gb_rec_scores['rmse']:8.1f}  MAE={gb_rec_scores['mae']:8.1f}  MAPE={gb_rec_scores['mape']:5.2f}%   (true 2-yr forecast)")

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 4.8))
ax.plot(design_fit.index[-80:], ytr.iloc[-80:], color=HUES['silver'], lw=1.2, label='Recent training')
ax.plot(idx, oos_wk, color=HUES['slate'], lw=2.2, label='Actual')
ax.plot(idx, gb_recursive, color=HUES['brick'], lw=2.0, label='GB recursive (multi-step)')
ax.plot(idx, gb_onestep, color=HUES['amber'], lw=1.4, ls=(0, (2, 2)), label='GB one-step (actual lag-1)')
ax.set(title='Gradient Boosting forecasts vs actual', ylabel='MW', xlabel='Date')
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

# feature importance drawn as a horizontal lollipop
imp = pd.Series(gb_engine.feature_importances_, index=PREDICTORS).sort_values()
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.hlines(imp.index, 0, imp.to_numpy(), color=HUES['silver'], lw=2)
ax.plot(imp.to_numpy(), imp.index, 'o', color=HUES['brick'], ms=9)
ax.set(title='Gradient Boosting feature importance', xlabel='Relative importance')
plt.tight_layout()
plt.show()

### Reading the Gradient Boosting results

The importance ranking is led by the lag features - last week's load and the value one
year earlier - which mirrors the strong annual cycle exposed by the decomposition; the
calendar and temperature columns act as refinements rather than primary drivers. The
recursive multi-step curve follows the seasonal envelope closely yet, like the
state-space models, cannot anticipate the 2020 demand shock. The booster's advantage is
that it learns non-linear interactions among the drivers automatically, without the
manual squared-temperature term the linear model needed. Its limitations here are the
lack of native prediction intervals and the way long recursive roll-outs compound their
own errors, which is exactly why it is reported alongside - never in place of - the
seasonal benchmark.

## 8. Hourly LSTM

### Background

Long Short-Term Memory (LSTM) networks are recurrent models with gated memory that
learn long temporal dependencies while side-stepping the vanishing-gradient problem
of plain RNNs. In load forecasting (e.g. Kong et al., 2017, *IEEE Transactions on
Smart Grid*) they are popular because they learn short-run fluctuations and longer
seasonal shape directly from data, without ARIMA's stationarity assumptions. Their
weaknesses are appetite for data and error build-up in long recursive forecasts,
which we quantify explicitly below.

We model the **hourly** series (168-hour lookback) and later aggregate to weekly for
comparison with the other models.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

hourly = de_power['mw'].copy()
squeezer = MinMaxScaler()
hourly_norm = squeezer.fit_transform(hourly.to_numpy().reshape(-1, 1))

SEQ_LEN = 168  # one week of hourly lags
feats, labels = [], []
for t in range(SEQ_LEN, len(hourly_norm)):
    feats.append(hourly_norm[t - SEQ_LEN:t, 0])
    labels.append(hourly_norm[t, 0])
feats = np.asarray(feats)
labels = np.asarray(labels)

hours_out = 24 * 7 * 52 * 2  # final two years, in hours
cut = len(feats) - hours_out
Xtr_seq = feats[:cut].reshape(-1, SEQ_LEN, 1)
ytr_seq = labels[:cut]
Xte_seq = feats[cut:].reshape(-1, SEQ_LEN, 1)
yte_seq = labels[cut:]
print('Sequence tensors -> train', Xtr_seq.shape, '| test', Xte_seq.shape)

### Architecture search

We compare five recurrent architectures that vary depth, width, dropout and batch
size, each trained with early stopping on the same split. We record validation loss
and hourly test RMSE and carry forward the best one.

In [ ]:
def assemble_net(layer_units, drop):
    net = Sequential()
    net.add(Input(shape=(SEQ_LEN, 1)))
    for k, u in enumerate(layer_units):
        net.add(LSTM(u, return_sequences=(k < len(layer_units) - 1)))
        net.add(Dropout(drop))
    net.add(Dense(24, activation='relu'))
    net.add(Dense(1))
    net.compile(optimizer='adam', loss='mse')
    return net


net_grid = [
    {'tag': 'compact-32', 'units': [32], 'drop': 0.15, 'batch': 128},
    {'tag': 'stacked-48-24', 'units': [48, 24], 'drop': 0.20, 'batch': 128},
    {'tag': 'stacked-96-48', 'units': [96, 48], 'drop': 0.30, 'batch': 64},
    {'tag': 'wide-64', 'units': [64], 'drop': 0.25, 'batch': 128},
    {'tag': 'deep-96-48-24', 'units': [96, 48, 24], 'drop': 0.30, 'batch': 64},
]

trials = []
for spec in net_grid:
    tf.random.set_seed(SEED)
    net = assemble_net(spec['units'], spec['drop'])
    stopper = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    hist = net.fit(Xtr_seq, ytr_seq, epochs=10, batch_size=spec['batch'],
                   validation_split=0.1, callbacks=[stopper], verbose=0)
    yhat = squeezer.inverse_transform(net.predict(Xte_seq, verbose=0)).ravel()
    ytrue = squeezer.inverse_transform(yte_seq.reshape(-1, 1)).ravel()
    trials.append({'Config': spec['tag'], 'Units': '-'.join(map(str, spec['units'])),
                   'Dropout': spec['drop'], 'Batch': spec['batch'],
                   'Val loss': round(min(hist.history['val_loss']), 6),
                   'Hourly RMSE': round(root_mse(ytrue, yhat), 2)})
trial_tbl = pd.DataFrame(trials).sort_values('Hourly RMSE').reset_index(drop=True)
print(trial_tbl.to_string(index=False))
top_spec = next(s for s in net_grid if s['tag'] == trial_tbl.iloc[0]['Config'])
print('\nBest architecture:', top_spec['tag'])

In [ ]:
tf.random.set_seed(SEED)
lstm_net = assemble_net(top_spec['units'], top_spec['drop'])
stopper = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
train_log = lstm_net.fit(Xtr_seq, ytr_seq, epochs=10, batch_size=top_spec['batch'],
                         validation_split=0.1, callbacks=[stopper], verbose=0)
print('Retrained final network:', top_spec['tag'])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_log.history['loss'], color=HUES['violet'], marker='o', ms=3, label='Train MSE')
ax.plot(train_log.history['val_loss'], color=HUES['brick'], marker='s', ms=3, label='Validation MSE')
ax.set(title='LSTM optimisation curve', xlabel='Epoch', ylabel='MSE (scaled)')
ax.legend()
plt.tight_layout()
plt.show()

### Rolling versus open-loop

A **rolling** (one-step) forecast feeds the network the *actual* previous 168 hours at
each step. An **open-loop** forecast feeds the network its own predictions, which is the
only honest way to cover the full two-year horizon from a single origin. Both are
reported and clearly labelled; comparing a one-step forecast with a multi-step one
would be misleading.

In [ ]:
# (1) rolling one-step: fed the ACTUAL previous 168 hours at each step.
roll_hat = squeezer.inverse_transform(lstm_net.predict(Xte_seq, batch_size=256, verbose=0)).ravel()
roll_true = squeezer.inverse_transform(yte_seq.reshape(-1, 1)).ravel()

# (2) open-loop recursive: predictions are fed back in (true multi-step).
seed_window = hourly_norm[cut:cut + SEQ_LEN, 0].copy()
loop_scaled = []
for s in range(hours_out):
    nxt = lstm_net.predict(seed_window.reshape(1, SEQ_LEN, 1), verbose=0)[0, 0]
    loop_scaled.append(nxt)
    seed_window = np.append(seed_window[1:], nxt)
    if (s + 1) % 2000 == 0:
        print(f'   open-loop progress {s + 1}/{hours_out} hours')
loop_hat = squeezer.inverse_transform(np.array(loop_scaled).reshape(-1, 1)).ravel()

hour_index = hourly.index[SEQ_LEN + cut:]
roll_rmse_hr = root_mse(roll_true, roll_hat)
loop_rmse_hr = root_mse(roll_true[:hours_out], loop_hat)

roll_hat_wk = pd.Series(roll_hat, index=hour_index).resample('W').mean()
roll_true_wk = pd.Series(roll_true, index=hour_index).resample('W').mean()
lstm_roll_wk_rmse = root_mse(roll_true_wk, roll_hat_wk)

loop_hat_wk = pd.Series(loop_hat, index=hour_index[:hours_out]).resample('W').mean()
loop_true_wk = pd.Series(roll_true[:hours_out], index=hour_index[:hours_out]).resample('W').mean()
lstm_loop_wk_rmse = root_mse(loop_true_wk, loop_hat_wk)

print(f'Rolling   RMSE hourly : {roll_rmse_hr:8.1f} MW   [one-step, sees actuals]')
print(f'Rolling   RMSE weekly : {lstm_roll_wk_rmse:8.1f} MW   [one-step, sees actuals]')
print(f'Open-loop RMSE weekly : {lstm_loop_wk_rmse:8.1f} MW   [true multi-step]')
print(f'Open-loop RMSE hourly : {loop_rmse_hr:8.1f} MW   [true multi-step]')

With the labels correct, the **rolling weekly** RMSE sits *below* the hourly rolling
RMSE because weekly averaging cancels independent hourly errors. The **open-loop**
weekly RMSE is far larger: recursive feedback accumulates error across two years, so a
univariate open-loop LSTM is not viable over this horizon without external drivers.

## 9. Answers to the assignment questions

**Q1 - Which models meaningfully beat the seasonal-replay benchmark?**
Comparisons are only fair within a forecast type. On multi-step weekly RMSE, **no model
beats the seasonal replay** in this run: the recursive Gradient Boosting model is the
closest challenger and essentially ties it, then SARIMAX, then pure SARIMA, while the
open-loop LSTM trails badly because of recursive drift. German weekly demand is
dominated by a stable annual cycle, so "repeat last year" is an unusually strong
two-year baseline. Two things reinforce this: the hold-out contains the 2020 COVID-19
dip (an exogenous shock none of the models anticipate, whereas the replay simply
reuses the pre-shock profile), and over long horizons SARIMA/SARIMAX revert toward
their mean with widening intervals. The one-step GB and rolling LSTM look far more
accurate only because they consume the actual previous value and must not be compared
with the multi-step baseline.

**Q2 - How was leakage avoided in the temperature features?**
Every lag/rolling feature references only the past (`temp_lag = shift(1)`, `temp_ma4 =
trailing rolling(4).mean()`, `load_lag = shift(1)`, `load_yearlag = shift(52)`).
No feature at week *t* uses week *t* or later. The current-week temperature/holiday are
treated as known under the explicit conditional-forecast assumption, and the recursive
GB forecast fills its own load lags with predictions past the origin, so no future
actual ever leaks into the multi-step result.

**Q3 - Justify the differencing orders and seasonal period.**
`d = 1`: ADF/KPSS confirm stationarity after one difference and `d = 2` over-differences.
`D = 1`, `s = 52`: the decomposition and ACF show a strong annual cycle in weekly data,
so one seasonal difference at lag 52 removes it. The seasonal `(P, Q)` were searched
over `{0, 1}`, and the non-seasonal `(p, q)` were chosen by within-`d` AIC plus a
parsimony rule.

**Q4 - Do the covariates help, and are they known at the origin?**
Adding temperature (with a squared term for the U-shaped curve), its lag and a holiday
flag lowers the SARIMAX RMSE relative to pure SARIMA. They are **not fully known** at the
origin: temperature needs a weather forecast (adding uncertainty) while holidays are
deterministic, so the reported figure is a conditional forecast.

**Q5 - Interpretability and complexity.**

| Aspect | SARIMAX | Gradient Boosting | LSTM |
|---|---|---|---|
| Interpretability | High (coefficients, intervals) | Medium (feature importance) | Low (black box) |
| Complexity | Low-medium | Medium (feature engineering) | High (architecture + GPU) |
| Uncertainty | Native intervals | Needs bootstrapping | No native support |
| Non-linearity | Manual (temp squared) | Native (boosted trees) | Native (learned) |
| Training cost | Minutes | Seconds to minutes | Minutes to hours |

**Q6 - Which model for operational use?**
We recommend **SARIMAX** - not because it is the most accurate (the seasonal replay and
recursive Gradient Boosting edge it out on RMSE here), but because it best satisfies the
operational criteria together: native confidence intervals for capacity and risk
planning, interpretable temperature/holiday coefficients for what-if analysis, support
for exogenous drivers, and cheap retraining. The seasonal replay gives no uncertainty
and no covariate insight; Gradient Boosting is competitive and captures non-linearity but
has no native intervals and is less transparent; the open-loop LSTM is unsuitable over
this long horizon. Any deployment should keep tracking the seasonal-replay benchmark and
retrain as new (post-shock) data arrives.

## 10. Consolidated comparison

Finally we gather RMSE, MAE and MAPE for every model with a **forecast-type** column so
like is compared with like. The lollipop chart contrasts the multi-step forecasts
against the seasonal-replay baseline; the overlay then shows the multi-step forecasts
against the hold-out, with the open-loop LSTM in its own panel so its recursive drift
does not squash the shared axis.

In [ ]:
def row_for(name, actual, predicted, kind):
    m = accuracy_metrics(actual, predicted)
    return {'Model': name, 'Type': kind, 'RMSE [MW]': round(m['rmse'], 1),
            'MAE [MW]': round(m['mae'], 1), 'MAPE [%]': round(m['mape'], 2)}


ledger = pd.DataFrame([
    row_for('Historical mean', oos_wk, flat_mean, 'multi-step'),
    row_for('Last value', oos_wk, carry_forward, 'multi-step'),
    row_for('Seasonal replay', oos_wk, seasonal_replay, 'multi-step'),
    row_for('Linear drift', oos_wk, linear_drift, 'multi-step'),
    row_for('SARIMA', oos_wk, sarima_fc_mean, 'multi-step'),
    row_for('SARIMAX (temp+holiday)', oos_wk, sarimax_fc_mean, 'multi-step (conditional)'),
    row_for('Gradient Boosting (recursive)', oos_wk, gb_recursive, 'multi-step (conditional)'),
    row_for('Gradient Boosting (one-step)', yte, gb_onestep, 'one-step (actual lag-1)'),
    row_for('LSTM (open-loop)', loop_true_wk, loop_hat_wk, 'multi-step'),
    row_for('LSTM (rolling)', roll_true_wk, roll_hat_wk, 'one-step (actual lag-1)'),
]).sort_values('RMSE [MW]').reset_index(drop=True)
base_rmse = ledger.loc[ledger['Model'] == 'Seasonal replay', 'RMSE [MW]'].iloc[0]
ledger['Gap vs replay'] = (ledger['RMSE [MW]'] - base_rmse).round(1)
print('Seasonal-replay weekly RMSE baseline:', base_rmse, 'MW\n')
print(ledger.to_string(index=False))

In [ ]:
multi = (ledger[ledger['Type'].str.startswith('multi-step')]
         .sort_values('RMSE [MW]').reset_index(drop=True))
champ = int(multi['RMSE [MW]'].idxmin())
dot_colors = [HUES['brick'] if i == champ else HUES['violet'] for i in range(len(multi))]

fig, ax = plt.subplots(figsize=(10.5, 5.2))
ax.hlines(multi['Model'], base_rmse, multi['RMSE [MW]'], color='lightgray', lw=1.6, zorder=1)
ax.scatter(multi['RMSE [MW]'], multi['Model'], color=dot_colors, s=130, zorder=3, edgecolor='black')
ax.axvline(base_rmse, color=HUES['slate'], ls='--', lw=1.2, label=f'Seasonal replay ({base_rmse:.0f} MW)')
for r, m in zip(multi['RMSE [MW]'], multi['Model']):
    ax.text(r, m, f'  {r:.0f}', va='center', fontsize=9)
ax.set(title='Multi-step accuracy - distance from the seasonal-replay line (lower is better)',
       xlabel='Weekly RMSE [MW]')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(fit_wk.index, fit_wk, color=HUES['silver'], lw=0.9, alpha=0.7, label='Train')
ax.plot(idx, oos_wk, color=HUES['slate'], lw=2.4, label='Actual (hold-out)')
ax.plot(idx, seasonal_replay, color=HUES['amber'], lw=1.4, ls=(0, (5, 2)), label='Seasonal replay')
ax.plot(idx, sarima_fc_mean, color=HUES['teal'], lw=1.4, label='SARIMA')
ax.plot(idx, sarimax_fc_mean, color=HUES['olive'], lw=1.4, label='SARIMAX')
ax.plot(idx, gb_recursive, color=HUES['brick'], lw=2.2, label='Gradient Boosting (recursive)')
y_low = min(fit_wk.min(), oos_wk.min()) * 0.9
y_high = max(fit_wk.max(), oos_wk.max()) * 1.1
ax.set_ylim(y_low, y_high)
ax.set(title='Multi-step forecasts vs actual (open-loop LSTM shown separately)',
       ylabel='MW', xlabel='Date')
ax.legend(ncol=3, loc='lower left')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(idx, oos_wk, color=HUES['slate'], lw=2.2, label='Actual (hold-out)')
ax.plot(loop_hat_wk.index, loop_hat_wk, color=HUES['brick'], lw=1.6, label='LSTM open-loop (recursive)')
ax.set(title='Open-loop LSTM drift across the two-year horizon (separate scale)',
       ylabel='MW', xlabel='Date')
ax.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()